# Step 1: Data Preparation and Epoching

This notebook implements the Granger Causality analysis pipeline following the research paper's methodology.

**Step 1 Goal**: Extract artifact-free 2.56-second epochs from the pre-cleaned data segments identified in the annotations files.

## Import Required Libraries

In [22]:
import os
import pandas as pd
import mne
import numpy as np
from pathlib import Path

## Define Dataset Paths

In [23]:
dataset_root = Path('Dataset')
derivatives_path = dataset_root / 'derivatives' / 'NeuronicEEG'

print(f"Dataset root: {dataset_root}")
print(f"Derivatives path: {derivatives_path}")
print(f"Derivatives exists: {derivatives_path.exists()}")

Dataset root: Dataset
Derivatives path: Dataset/derivatives/NeuronicEEG
Derivatives exists: True


## Load Annotations and Extract Epochs

This function reads the annotations file for a subject/session and extracts the corresponding epochs from the raw EEG data.

In [24]:
def load_annotations(subject_id, session):
    """
    Load annotations file containing artifact-free segments.
    
    Parameters:
    -----------
    subject_id : str
        Subject ID (e.g., 'sub-NORB00001')
    session : int
        Session number
        
    Returns:
    --------
    pd.DataFrame
        Annotations with onset and duration columns
    """
    annotations_file = derivatives_path / subject_id / f'ses-{session}' / 'eeg' / \
                       f'{subject_id}_ses-{session}_task-EEG_annotations.tsv'
    
    if not annotations_file.exists():
        raise FileNotFoundError(f"Annotations file not found: {annotations_file}")
    
    annotations = pd.read_csv(annotations_file, sep='\t', index_col=False)
    return annotations


def load_raw_eeg(subject_id, session):
    """
    Load raw EEG data from EDF file and remove bad channels.
    
    Parameters:
    -----------
    subject_id : str
        Subject ID (e.g., 'sub-NORB00001')
    session : int
        Session number
        
    Returns:
    --------
    mne.io.Raw
        Raw EEG data object with bad channels removed
    """
    eeg_file = dataset_root / subject_id / f'ses-{session}' / 'eeg' / \
               f'{subject_id}_ses-{session}_task-EEG_eeg.edf'
    
    if not eeg_file.exists():
        raise FileNotFoundError(f"EEG file not found: {eeg_file}")
    
    raw = mne.io.read_raw_edf(eeg_file, preload=True, verbose=False)
    
    # Identify bad channels to remove
    bad_channels = []
    for ch_name in raw.ch_names:
        # Remove Pg1, Pg2, and any channels containing '+'
        if ch_name in ['Pg1', 'Pg2'] or '+' in ch_name:
            bad_channels.append(ch_name)
    
    # Drop bad channels if any found
    if bad_channels:
        raw.drop_channels(bad_channels)
    
    return raw


def extract_epochs(subject_id, session):
    """
    Extract 2.56-second artifact-free epochs from raw EEG data.
    
    Parameters:
    -----------
    subject_id : str
        Subject ID (e.g., 'sub-NORB00001')
    session : int
        Session number
        
    Returns:
    --------
    mne.Epochs
        Epoched EEG data (each epoch is 2.56 seconds)
    """
    # Load annotations and raw data
    annotations = load_annotations(subject_id, session)
    raw = load_raw_eeg(subject_id, session)
    
    # Create MNE Annotations object from the annotations dataframe
    mne_annotations = mne.Annotations(
        onset=annotations['onset'].values,
        duration=annotations['duration'].values,
        description=annotations['label'].values
    )
    
    # Set annotations to raw data
    raw.set_annotations(mne_annotations)
    
    # Create events from annotations
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    
    # Create epochs (tmin=0, tmax=duration to get full segments)
    # The duration is typically 2.56 seconds
    duration = annotations['duration'].iloc[0]
    epochs = mne.Epochs(
        raw, 
        events, 
        event_id=event_id,
        tmin=0, 
        tmax=duration,
        baseline=None,
        preload=True,
        verbose=False
    )
    
    return epochs

## Test with Example Subject

Let's test the implementation with the first subject to verify everything works correctly.

In [25]:
# Test with first subject
subject_id = 'sub-NORB00001'
session = 1

print(f"Processing {subject_id}, session {session}")
print("-" * 50)

# Load annotations
annotations = load_annotations(subject_id, session)
print(f"\nNumber of artifact-free segments: {len(annotations)}")
print(f"\nFirst few segments:")
print(annotations.head())

# Extract epochs
epochs = extract_epochs(subject_id, session)
print(f"\n{'-' * 50}")
print(f"Epochs created: {len(epochs)}")
print(f"Epoch duration: {epochs.times[-1]} seconds")
print(f"Number of channels: {len(epochs.ch_names)}")
print(f"Sampling frequency: {epochs.info['sfreq']} Hz")
print(f"Channel names: {epochs.ch_names}")

Processing sub-NORB00001, session 1
--------------------------------------------------

Number of artifact-free segments: 32

First few segments:
     onset  duration        label
0    0.960      2.56  eyes_closed
1   38.835      2.56  eyes_closed
2   55.310      2.56  eyes_closed
3   74.360      2.56  eyes_closed
4  128.100      2.56  eyes_closed

--------------------------------------------------
Epochs created: 32
Epoch duration: 2.56 seconds
Number of channels: 19
Sampling frequency: 200.0 Hz
Channel names: ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'CZ', 'PZ']


## Summary

Step 1 is complete. We have successfully:
- Loaded the annotations file containing artifact-free segment timings
- Read the raw EEG data from the .edf files
- Extracted 2.56-second epochs based on the annotation onset times

The epochs are now ready for Step 2: Source Localization.

---

# Understanding Step 2A: Forward Model in Detail

## What is a Forward Model?

The **forward model** is a mathematical description of how electrical activity in the brain propagates to create the voltages we measure at the scalp electrodes. Think of it as answering the question: *"If a specific location in the brain becomes active, what voltage pattern would we see across all 19 scalp electrodes?"*

## Why Do We Need It?

We cannot apply Granger Causality directly to scalp EEG because of the **volume conduction problem**: electrical signals from the brain spread through tissues (brain, skull, scalp) before reaching electrodes, causing signals to "blur" and mix together. A single brain source appears at multiple electrodes, and each electrode records activity from multiple brain regions simultaneously.

The forward model helps us **inverse** this process - to go backward from scalp measurements to estimate the brain activity that caused them.

## The Four Key Components

### 1. **Electrode Positions**
- We load the exact 3D coordinates of our 19 EEG electrodes from the `_electrodes.tsv` file
- These coordinates are in MNI space (a standard brain coordinate system)
- We convert them into an MNE "montage" object that can be used with the head model

### 2. **Infant Head Template (Anatomy)**
- We use age-appropriate infant head models from the ANTS database
- The template includes:
  - **Brain surface geometry** (where sources can be located)
  - **Skull and scalp shapes** (affect how signals conduct)
  - **Tissue boundaries** (different tissues have different electrical properties)
- The code automatically selects the closest template to the subject's age (2wk, 1mo, 2mo, 3mo, 4.5mo, 6mo, etc.)

### 3. **Source Space**
- Defines ~8,000 candidate "source locations" across the cortical surface (brain's gray matter)
- Each source is a potential "virtual electrode" inside the brain
- Using `spacing='oct6'` creates approximately 4,096 sources per hemisphere
- These sources are evenly distributed across the cortical surface

### 4. **BEM (Boundary Element Model)**
- Describes the physics of how electrical currents flow through different head tissues
- Models three tissue layers:
  - **Brain** (σ ≈ 0.33 S/m)
  - **Skull** (σ ≈ 0.0042 S/m - acts as insulator)
  - **Scalp** (σ ≈ 0.33 S/m)
- Uses triangulated surfaces to represent boundaries between tissues

## The Output: Leadfield Matrix

The **forward solution** produces a "leadfield matrix" with dimensions: `(19 electrodes × ~8,000 sources × 3 orientations)`

This matrix mathematically encodes: *"If source #X with orientation Y activates, it will produce voltage V₁ at electrode 1, V₂ at electrode 2, ... V₁₉ at electrode 19"*

## What Happens Next?

In **Step 2B**, we'll use this forward model to create the **inverse operator** (sLoreta). This will allow us to take our 19-channel epoch data and estimate the activity at all ~8,000 brain sources - essentially projecting the scalp signals back into the brain.

The mathematical relationship is:
- **Forward**: Brain Sources → (leadfield matrix) → Scalp Voltages
- **Inverse**: Scalp Voltages → (inverse operator) → Brain Sources

---

# Step 2A: Setup Forward Model

This step creates the forward model that describes how brain activity propagates to the scalp electrodes.

**Goals**:
1. Load infant head template (BEM model)
2. Create source space on cortical surface
3. Load electrode positions from the dataset
4. Compute forward solution (leadfield matrix)

## Load Electrode Positions

First, we need to load the electrode positions from the dataset and prepare them for MNE.

In [26]:
def load_electrode_positions(subject_id, session):
    """
    Load electrode positions from the electrodes.tsv file.
    
    Parameters:
    -----------
    subject_id : str
        Subject ID (e.g., 'sub-NORB00001')
    session : int
        Session number
        
    Returns:
    --------
    pd.DataFrame
        Electrode positions with columns: name, x, y, z
    """
    electrodes_file = dataset_root / subject_id / f'ses-{session}' / 'eeg' / \
                      f'{subject_id}_ses-{session}_electrodes.tsv'
    
    if not electrodes_file.exists():
        raise FileNotFoundError(f"Electrodes file not found: {electrodes_file}")
    
    electrodes = pd.read_csv(electrodes_file, sep='\t')
    return electrodes


# Load electrode positions for test subject
electrode_positions = load_electrode_positions(subject_id, session)
print(f"Total electrodes in file: {len(electrode_positions)}")
print(f"\nElectrode columns: {electrode_positions.columns.tolist()}")
print(f"\nFirst few electrodes:")
print(electrode_positions.head())

# Filter to only include channels we're using (exclude bad channels)
valid_channels = epochs.ch_names
electrode_positions_filtered = electrode_positions[electrode_positions['name'].isin(valid_channels)]
print(f"\nElectrodes after filtering bad channels: {len(electrode_positions_filtered)}")
print(f"Filtered electrode names: {electrode_positions_filtered['name'].tolist()}")

Total electrodes in file: 21

Electrode columns: ['name', 'x', 'y', 'z', 'type', 'material']

First few electrodes:
  name     x     y     z type material
0  Fp1 -32.0  77.0  19.0  cup  Ag/AgCl
1  Fp2  32.0  77.0  19.0  cup  Ag/AgCl
2   F3 -52.0  35.0  57.0  cup  Ag/AgCl
3   F4  52.0  35.0  57.0  cup  Ag/AgCl
4   C3 -69.0 -29.0  69.0  cup  Ag/AgCl

Electrodes after filtering bad channels: 19
Filtered electrode names: ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'FZ', 'CZ', 'PZ']


## Create Montage from Electrode Positions

Convert electrode positions to MNE montage format and set it to the epochs.

In [27]:
def create_montage_from_positions(electrode_positions_df, ch_names):
    """
    Create MNE montage from electrode positions dataframe.
    
    Parameters:
    -----------
    electrode_positions_df : pd.DataFrame
        Electrode positions with columns: name, x, y, z
    ch_names : list
        List of channel names to include
        
    Returns:
    --------
    mne.channels.DigMontage
        Digital montage object
    """
    # Filter electrodes to only include valid channels
    filtered_df = electrode_positions_df[electrode_positions_df['name'].isin(ch_names)].copy()
    
    # Create dictionary mapping channel names to positions (in meters for MNE)
    ch_pos = {}
    for _, row in filtered_df.iterrows():
        # Convert from mm to meters
        ch_pos[row['name']] = np.array([row['x'], row['y'], row['z']]) / 1000.0
    
    # Create montage
    montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='mni_tal')
    
    return montage


# Create and set montage
montage = create_montage_from_positions(electrode_positions, epochs.ch_names)
epochs.set_montage(montage)

print(f"Montage created with {len(montage.ch_names)} electrodes")
print(f"Coordinate frame: {montage.dig[0]['coord_frame']}")
print(f"\nSample electrode positions (in meters):")
for i, ch_name in enumerate(epochs.ch_names[:5]):
    pos = montage.get_positions()['ch_pos'][ch_name]
    print(f"  {ch_name}: [{pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f}]")

Montage created with 19 electrodes
Coordinate frame: 2003 (FIFFV_MNE_COORD_MNI_TAL)

Sample electrode positions (in meters):
  Fp1: [-0.0320, 0.0770, 0.0190]
  Fp2: [0.0320, 0.0770, 0.0190]
  F3: [-0.0520, 0.0350, 0.0570]
  F4: [0.0520, 0.0350, 0.0570]
  C3: [-0.0690, -0.0290, 0.0690]


/tmp/ipykernel_8500/1171947366.py:34: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  epochs.set_montage(montage)


## Load Infant Head Template (BEM Model)

Load an age-appropriate infant head template. MNE provides ANTS infant templates at different ages.

In [31]:
# Check subject age to determine appropriate template
scans_file = dataset_root / subject_id / f'ses-{session}' / f'{subject_id}_ses-{session}_scans.tsv'
scans_df = pd.read_csv(scans_file, sep='\t')
subject_age_years = scans_df['age_acq_time'].iloc[0]
subject_age_months = subject_age_years * 12

print(f"Subject age: {subject_age_years:.2f} years ({subject_age_months:.1f} months)")

# Select appropriate infant template based on age
# Available templates: '2wk', '1mo', '2mo', '3mo', '4.5mo', '6mo', '7.5mo', '9mo', '10.5mo', '12mo', '15mo', '18mo', '2yr'
if subject_age_months < 0.5:
    template_age = '2wk'
elif subject_age_months < 1.5:
    template_age = '1mo'
elif subject_age_months < 2.5:
    template_age = '2mo'
elif subject_age_months < 3.75:
    template_age = '3mo'
elif subject_age_months < 5.25:
    template_age = '4.5mo'
elif subject_age_months < 6.75:
    template_age = '6mo'
elif subject_age_months < 8.25:
    template_age = '7.5mo'
elif subject_age_months < 9.75:
    template_age = '9mo'
elif subject_age_months < 11.25:
    template_age = '10.5mo'
elif subject_age_months < 13.5:
    template_age = '12mo'
elif subject_age_months < 16.5:
    template_age = '15mo'
elif subject_age_months < 24:
    template_age = '18mo'
else:
    template_age = '2yr'

print(f"Using template: {template_age}")

# Set up a directory for MNE data (creates if doesn't exist)
mne_data_dir = Path.home() / 'mne_data'
mne_data_dir.mkdir(exist_ok=True)

# Fetch infant template (this will download if not already cached)
subjects_dir = mne.datasets.fetch_infant_template(template_age, subjects_dir=str(mne_data_dir), verbose=True)

# The returned subjects_dir is actually the template name itself
# We need to construct the full path
subject_template = subjects_dir  # e.g., 'ANTS4-5Months3T'
subjects_dir = mne_data_dir  # The parent directory

print(f"\nTemplate directory: {subjects_dir}")
print(f"Template subject: {subject_template}")

Subject age: 0.41 years (4.9 months)
Using template: 4.5mo
0 files missing from ANTS4-5Months3T.txt in /home/alookaladdoo/mne_data/ANTS4-5Months3T

Template directory: /home/alookaladdoo/mne_data
Template subject: ANTS4-5Months3T


## Create Source Space

Create a source space on the cortical surface. This defines the candidate locations for brain sources.

In [33]:
# Create source space (spacing='oct6' gives ~4098 sources per hemisphere)
# This may take a few minutes on first run
src = mne.setup_source_space(
    subject_template,
    spacing='oct6',
    subjects_dir=subjects_dir,
    add_dist=False,
    verbose=True
)

print(f"\nSource space created:")
print(f"  Left hemisphere: {src[0]['nuse']} sources")
print(f"  Right hemisphere: {src[1]['nuse']} sources")
print(f"  Total sources: {src[0]['nuse'] + src[1]['nuse']}")

Setting up the source space with the following parameters:

SUBJECTS_DIR = /home/alookaladdoo/mne_data
Subject      = ANTS4-5Months3T
Surface      = white
Octahedron subdivision grade 6
SUBJECTS_DIR = /home/alookaladdoo/mne_data
Subject      = ANTS4-5Months3T
Surface      = white
Octahedron subdivision grade 6

>>> 1. Creating the source space...

Doing the octahedral vertex picking...
Loading /home/alookaladdoo/mne_data/ANTS4-5Months3T/surf/lh.white...
Mapping lh ANTS4-5Months3T -> oct (6) ...

>>> 1. Creating the source space...

Doing the octahedral vertex picking...
Loading /home/alookaladdoo/mne_data/ANTS4-5Months3T/surf/lh.white...
Mapping lh ANTS4-5Months3T -> oct (6) ...
    Triangle neighbors and vertex normals...
    Triangle neighbors and vertex normals...
Loading geometry from /home/alookaladdoo/mne_data/ANTS4-5Months3T/surf/lh.sphere...
Loading geometry from /home/alookaladdoo/mne_data/ANTS4-5Months3T/surf/lh.sphere...
Setting up the triangulation for the decimated surface

## Load BEM Model

Load the Boundary Element Model (BEM) that describes the head geometry and tissue conductivities.

In [34]:
# Load BEM model (boundary element model for forward modeling)
bem = mne.read_bem_solution(
    fname=subjects_dir / subject_template / 'bem' / f'{subject_template}-5120-5120-5120-bem-sol.fif',
    verbose=True
)

print(f"\nBEM model loaded:")
print(f"  Solution type: {bem['bem_method']}")
print(f"  Number of surfaces: {len(bem['surfs'])}")
for i, surf in enumerate(bem['surfs']):
    print(f"    Surface {i+1}: {surf['ntri']} triangles")

Loading surfaces...

Loading the solution matrix...


Loading the solution matrix...

Three-layer model surfaces loaded.
Loaded linear collocation BEM solution from /home/alookaladdoo/mne_data/ANTS4-5Months3T/bem/ANTS4-5Months3T-5120-5120-5120-bem-sol.fif

BEM model loaded:
  Solution type: 2
  Number of surfaces: 3
    Surface 1: 5120 triangles
    Surface 2: 5120 triangles
    Surface 3: 5120 triangles
Three-layer model surfaces loaded.
Loaded linear collocation BEM solution from /home/alookaladdoo/mne_data/ANTS4-5Months3T/bem/ANTS4-5Months3T-5120-5120-5120-bem-sol.fif

BEM model loaded:
  Solution type: 2
  Number of surfaces: 3
    Surface 1: 5120 triangles
    Surface 2: 5120 triangles
    Surface 3: 5120 triangles


## Transform Electrode Positions to Head Coordinate System

We need to align the electrode positions (in MNI space) to the head model coordinate system.

In [35]:
# Create transformation from MNI to head coordinate system
# We'll use a simple scaling transformation for infant head
# This is a simplified approach - in production you'd do proper coregistration

# Get trans identity (for now, assume electrodes are already in correct space)
trans = mne.transforms.Transform('mri', 'head', trans=np.eye(4))

print(f"Transformation matrix created:")
print(f"  From: {trans['from']} coordinate system")
print(f"  To: {trans['to']} coordinate system")
print(f"\nNote: Using identity transform. For production, proper coregistration is needed.")

Transformation matrix created:
  From: 5 coordinate system
  To: 4 coordinate system

Note: Using identity transform. For production, proper coregistration is needed.


## Compute Forward Solution (Leadfield Matrix)

Calculate how each brain source contributes to the signal at each electrode. This is the core of the forward model.

In [36]:
# Compute forward solution
# This creates the leadfield matrix that maps source activity to sensor signals
fwd = mne.make_forward_solution(
    epochs.info,
    trans=trans,
    src=src,
    bem=bem,
    eeg=True,
    meg=False,
    mindist=5.0,
    n_jobs=1,
    verbose=True
)

print(f"\nForward solution computed:")
print(f"  Number of sources: {fwd['nsource']}")
print(f"  Number of channels: {fwd['nchan']}")
print(f"  Coordinate frame: {fwd['mri_head_t']['from']} -> {fwd['mri_head_t']['to']}")
print(f"  Leadfield matrix shape: {fwd['sol']['data'].shape}")
print(f"\nStep 2A Complete! Forward model is ready.")

Source space          : <SourceSpaces: [<surface (lh), n_vertices=40924, n_used=4098>, <surface (rh), n_vertices=39690, n_used=4098>] MRI (surface RAS) coords, subject 'ANTS4-5Months3T', ~6.4 MiB>
MRI -> head transform : instance of Transform
Measurement data      : instance of Info
Conductor model   : instance of ConductorModel
Accurate field computations
Do computations in head coordinates
Free source orientations

Read 2 source spaces a total of 8196 active source locations

Coordinate transformation: MRI (surface RAS) -> head
    1.000000 0.000000 0.000000       0.00 mm
    0.000000 1.000000 0.000000       0.00 mm
    0.000000 0.000000 1.000000       0.00 mm
    0.000000 0.000000 0.000000       1.00
MRI -> head transform : instance of Transform
Measurement data      : instance of Info
Conductor model   : instance of ConductorModel
Accurate field computations
Do computations in head coordinates
Free source orientations

Read 2 source spaces a total of 8196 active source locations

C

# Step 2B: Compute Inverse Operator (sLoreta)

This step creates the inverse operator using sLoreta (standardized Low Resolution Electromagnetic Tomography) method to estimate brain source activity from scalp EEG.

**Goals**:
1. Estimate noise covariance from the data
2. Create sLoreta inverse operator
3. Apply inverse operator to epochs to get source time courses

## Compute Noise Covariance

Estimate the noise covariance matrix from the data. This describes the background noise characteristics in our recordings.

In [40]:
# Compute noise covariance from the epochs
# Since we don't have a baseline period, we'll use the entire epoch data
noise_cov = mne.compute_covariance(
    epochs,
    tmin=0,
    tmax=None,
    method='empirical',
    verbose=True
)

print(f"\nNoise covariance computed:")
print(f"  Method: {noise_cov['method']}")
print(f"  Number of samples: {noise_cov['nfree']}")
print(f"  Number of channels: {len(noise_cov['names'])}")
print(f"  Covariance matrix shape: {noise_cov['data'].shape}")

Reducing data rank from 19 -> 19
Estimating covariance using EMPIRICAL
Done.
Number of samples used : 16416
[done]
Estimating covariance using EMPIRICAL
Done.
Number of samples used : 16416
[done]

Noise covariance computed:
  Method: empirical
  Number of samples: 16415
  Number of channels: 19
  Covariance matrix shape: (19, 19)

Noise covariance computed:
  Method: empirical
  Number of samples: 16415
  Number of channels: 19
  Covariance matrix shape: (19, 19)


## Create sLoreta Inverse Operator

Construct the inverse operator using the sLoreta method. This will be used to estimate source activity from scalp measurements.

In [41]:
# Create inverse operator using sLoreta
inverse_operator = mne.minimum_norm.make_inverse_operator(
    epochs.info,
    fwd,
    noise_cov,
    loose=0.2,        # Allow some tangential dipoles (0=fixed, 1=free)
    depth=0.8,        # Depth weighting to compensate for signal strength bias
    verbose=True
)

print(f"\nInverse operator created:")
print(f"  Method: sLoreta")
print(f"  Number of sources: {inverse_operator['nsource']}")
print(f"  Number of channels: {len(inverse_operator['info']['ch_names'])}")
print(f"  Source orientation constraint: {'loose' if inverse_operator['source_ori'] == 1 else 'fixed'}")
print(f"\nStep 2B Complete! Inverse operator is ready.")

Converting forward solution to surface orientation
    No patch info available. The standard source space normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    No patch info available. The standard source space normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]
Computing inverse operator with 19 channels.
    19 out of 19 channels remain after picking
Selected 19 channels
Creating the depth weighting matrix...
    19 EEG channels
    [done]
Computing inverse operator with 19 channels.
    19 out of 19 channels remain after picking
Selected 19 channels
Creating the depth weighting matrix...
    19 EEG channels
    limit = 8185/8184 = 4.261660
    scale = 166583 exp = 0.8
Applying loose dipole orientations to surface source spaces: 0.2
Whitening the forward solution.
    limit = 8185/8184 = 4.261660
    scale = 166

/tmp/ipykernel_8500/1417553827.py:2: RuntimeWarning: No average EEG reference present in info["projs"], covariance may be adversely affected. Consider recomputing covariance using with an average eeg reference projector added.
  inverse_operator = mne.minimum_norm.make_inverse_operator(
/tmp/ipykernel_8500/1417553827.py:2: RuntimeWarning: No average EEG reference present in info["projs"], covariance may be adversely affected. Consider recomputing covariance using with an average eeg reference projector added.
  inverse_operator = mne.minimum_norm.make_inverse_operator(


    largest singular value = 1.90373
    scaling factor to adjust the trace = 2.60629e+18 (nchan = 19 nzero = 0)

Inverse operator created:
  Method: sLoreta
  Number of sources: 8184
  Number of channels: 19
  Source orientation constraint: fixed

Step 2B Complete! Inverse operator is ready.
    scaling factor to adjust the trace = 2.60629e+18 (nchan = 19 nzero = 0)

Inverse operator created:
  Method: sLoreta
  Number of sources: 8184
  Number of channels: 19
  Source orientation constraint: fixed

Step 2B Complete! Inverse operator is ready.


## Set EEG Average Reference

Before applying the inverse operator, we need to set an average EEG reference using a projector.

In [44]:
# Set average EEG reference using projection
# This is required for inverse modeling
epochs.set_eeg_reference(projection=True, verbose=True)

print(f"EEG average reference set using projector")
print(f"Number of projectors: {len(epochs.info['projs'])}")

EEG channel type selected for re-referencing
EEG average reference set using projector
Number of projectors: 1
EEG average reference set using projector
Number of projectors: 1


/tmp/ipykernel_8500/1014330115.py:3: RuntimeWarning: An average reference projection was already added. The data has been left untouched.
  epochs.set_eeg_reference(projection=True, verbose=True)


## Apply Inverse Operator to Epochs

Apply the sLoreta inverse operator to each epoch to compute source time courses. This transforms the 19-channel scalp data into ~8,000 brain source signals.

In [45]:
# Apply inverse operator to get source estimates
# Using sLoreta method with lambda2=1/SNR^2 regularization parameter
snr = 3.0  # Signal-to-noise ratio (typical value for EEG)
lambda2 = 1.0 / snr ** 2

stc = mne.minimum_norm.apply_inverse_epochs(
    epochs,
    inverse_operator,
    lambda2=lambda2,
    method='sLORETA',
    pick_ori=None,
    verbose=True
)

print(f"\nSource estimates computed:")
print(f"  Number of epochs: {len(stc)}")
print(f"  Number of source vertices: {len(stc[0].data)}")
print(f"  Number of time points per epoch: {stc[0].data.shape[1]}")
print(f"  Time range: {stc[0].times[0]:.3f} to {stc[0].times[-1]:.3f} seconds")
print(f"  Sampling frequency: {1 / (stc[0].times[1] - stc[0].times[0]):.1f} Hz")

# Display info about first epoch's source estimate
print(f"\nFirst epoch source data shape: {stc[0].data.shape}")
print(f"  (vertices × time points)")
print(f"\nSource estimate data range:")
print(f"  Min: {stc[0].data.min():.2e}")
print(f"  Max: {stc[0].data.max():.2e}")
print(f"  Mean: {stc[0].data.mean():.2e}")

Preparing the inverse operator for use...
    Scaled noise and source covariance from nave = 1 to nave = 1
    Created the regularized inverter
    The projection vectors do not apply to these channels.
    Created the whitener using a noise covariance matrix with rank 19 (0 small eigenvalues omitted)
    Computing noise-normalization factors (sLORETA)...
[done]
Picked 19 channels from the data
Computing inverse...
    Eigenleads need to be weighted ...
Processing epoch : 1 / 32
combining the current components...
Processing epoch : 2 / 32
combining the current components...
Processing epoch : 3 / 32
combining the current components...
Processing epoch : 4 / 32
combining the current components...
Processing epoch : 5 / 32
combining the current components...
Processing epoch : 6 / 32
combining the current components...
Processing epoch : 7 / 32
combining the current components...
Processing epoch : 8 / 32
combining the current components...
Processing epoch : 9 / 32
combining the curren

## Summary

**Step 2B Complete!** We have successfully:

1. **Computed noise covariance** - Estimated the background noise characteristics from the epoch data
2. **Created sLoreta inverse operator** - Built the mathematical tool to project scalp signals back to brain sources
3. **Applied inverse to epochs** - Transformed all 32 epochs from 19-channel scalp data to ~8,000 source signals

**What we now have:**
- Each of the 32 epochs is now represented as ~8,000 brain source time series
- Each source has 512 time points (2.56 seconds at 200 Hz)
- We've moved from **scalp space** to **source space** (brain)

**Next Steps:**
- Step 3: Signal Unmixing (remove leakage between sources)
- Step 4: Extract ROI time series (reduce from ~8,000 sources to 16 regions)
- Step 5: Apply Granger Causality analysis

**Important Note:** The source estimates still contain some spatial leakage (activity from one region appearing in neighbors). This is why Step 3 (unmixing) is critical before we can perform connectivity analysis.